# Neo su mini-AGI (esperimento)
Addestra un modello **byte-level da zero** con [volotat/mini-AGI](https://github.com/volotat/mini-AGI) sul dataset di NEO (azioni di scrittura stile Brave Leo).

⚠️ mini-AGI **non è un fine-tuner**: non parte da Qwen/Llama, impara dai byte (richiede GPU CUDA, ~8 GB). Con ~500-2000 esempi non otterrai la qualità di Micro-Neo (`neo_finetune_v5.ipynb`, LoRA su Qwen2.5): è un esperimento di *continual learning*. Per il modello da usare in NEO nel browser resta la v5.

Runtime: GPU (T4 va bene per prove).

### 1. Installazione

In [ ]:
!git clone -q https://github.com/volotat/mini-AGI
%cd mini-AGI
!pip install -q numpy pyyaml matplotlib flask chess zstandard datasets scipy
!python train.py read --help | head -40

### 2. Dataset NEO → formato chat di mini-AGI

In [ ]:
import json, os, random, subprocess
random.seed(0)
PASSES, LR = 3, 5e-5          # passate sul corpus; lr predefinito di mini-AGI
VAL_FRAC = 0.1                # quota di testi sorgente (sid) tenuta fuori

# stessi prompt di NEO (cella «Prompt condivisi» di neo_finetune_v5.ipynb)
SYSTEM = ("Sei un assistente di scrittura che lavora in locale. Rispondi SOLO con il risultato richiesto, "
          "senza premesse, commenti o virgolette di contorno. Usa la stessa lingua del testo dell'utente, "
          "salvo diversa richiesta. Non inventare fatti che non sono nel testo.")
TONES = {"professional":"professionale e curato","casual":"informale e colloquiale","enthusiastic":"entusiasta e positivo",
         "informative":"informativo e neutro","creative":"creativo e vivace"}
PLATFORMS = {"linkedin":"un post LinkedIn (tono professionale, 3-5 frasi, massimo 3 hashtag)",
             "x":"un post per X/Twitter (massimo 280 caratteri, massimo 2 hashtag)",
             "instagram":"una didascalia Instagram (tono coinvolgente, qualche emoji, 5-8 hashtag in fondo)",
             "facebook":"un post Facebook (tono amichevole, 2-4 frasi)"}
def build_prompt(a, p, text):
    ins = {"summary":"Riassumi il testo seguente in modo chiaro e conciso (un paragrafo breve oppure al massimo 5 punti elenco).",
           "explain":"Spiega il testo seguente in parole semplici, come a chi non conosce l'argomento. Chiarisci i termini difficili.",
           "paraphrase":"Parafrasa il testo seguente con parole diverse, mantenendo lo stesso significato e la stessa lunghezza circa.",
           "improve":"Migliora il testo seguente: correggi errori e refusi e rendi lo stile più chiaro e scorrevole, senza cambiare il significato.",
           "quote":"Dal testo seguente ricava una citazione breve e memorabile (una o due frasi) che ne catturi l'idea centrale.",
           "translate":"Traduci il testo seguente in inglese, mantenendo lo stile originale."}.get(a)
    if a=="tone": ins=f"Riscrivi il testo seguente con un tono {TONES[p]}, mantenendo il significato."
    if a=="length": ins = ("Riscrivi il testo seguente in forma molto più breve, tenendo solo le informazioni essenziali." if p=="shorter"
                           else "Riscrivi il testo seguente in forma più lunga e sviluppata, aggiungendo chiarezza e fluidità ma nessun fatto nuovo.")
    if a=="social": ins=f"Crea {PLATFORMS[p]} basato sul testo seguente."
    return f'{ins}\n\nTESTO:\n"""\n{text}\n"""'

# dataset: dal repo NEO (o carica a mano neo_dataset.jsonl / leo_dataset.jsonl)
if not os.path.exists("neo.jsonl"):
    subprocess.run("git clone -q --depth 1 https://github.com/scobru/neo /tmp/neo && cp /tmp/neo/dataset/leo_dataset.jsonl neo.jsonl", shell=True)
if not os.path.exists("neo.jsonl"):
    from google.colab import files; up = files.upload(); os.rename(next(iter(up)), "neo.jsonl")
rows = [json.loads(l) for l in open("neo.jsonl", encoding="utf8") if l.strip()]

sids = sorted({r["sid"] for r in rows}); random.shuffle(sids)
val_sids = set(sids[:max(1, int(len(sids)*VAL_FRAC))])
def fmt(r):
    return f"<user>\n{SYSTEM}\n\n{build_prompt(r['action'], r['param'], r['input'])}\n</user>\n<bot>\n{r['output']}\n</bot>\n"
for d in ("corpus/train", "corpus/val"): os.makedirs(d, exist_ok=True)
tr = [r for r in rows if r["sid"] not in val_sids]; va = [r for r in rows if r["sid"] in val_sids]
for name, part in (("train", tr), ("val", va)):
    random.shuffle(part)
    open(f"corpus/{name}/neo.txt", "w", encoding="utf8").write("".join(fmt(r) for r in part))
print(len(tr), "train |", len(va), "val |", os.path.getsize("corpus/train/neo.txt")//1024, "KB")

### 3. Training (`train.py read`: leggere = addestrare)

In [ ]:
!python train.py read corpus/train --passes {PASSES} --lr {LR} --save --weights-dir weights --held-out corpus/val

### 4. Prova il modello
mini-AGI non ha una CLI di generazione: usa la web UI. Scrivi nel formato `<user> … </user>` e il modello risponde in `<bot>`.

In [ ]:
import subprocess, time
srv = subprocess.Popen(["python", "serve.py", "--port", "8080"])
time.sleep(10)
from google.colab import output
output.serve_kernel_port_as_window(8080)

### 5. Salva i pesi (la cartella `weights/` *è* il modello)

In [ ]:
!zip -qr neo_miniagi_weights.zip weights
from google.colab import files
files.download("neo_miniagi_weights.zip")